In [ ]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import pandas as pd  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import skimage as ski
import xarray as xr
import sys

sys.path.append("../..")  # Adds higher directory to python modules path.
from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from src import SR_Functions

SR_F = SR_Functions.SuperRes_Functions()

import gaussoptfuncs
from numba import jit


import types

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [ ]:
# data_folder = '/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/old/'
data_folder = "../Camera_Calibrations/Ximea_Camera/"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])
image_size = 16
camera_parameters = {}
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
camera_parameters["gain"] = np.full((image_size, image_size), np.median(gain))
camera_parameters["offset"] = np.full((image_size, image_size), np.median(offset))
camera_parameters["variance"] = np.full((image_size, image_size), 1e-12)
camera_parameters["readnoise"] = np.full((image_size, image_size), 1e-12)
camera_parameters["rqe"] = np.full((image_size, image_size), 1.0)
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ["B", "G", "R"]
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks

In [ ]:
notch_filter = "semrock-nf03-405-488-561-635e"
dichroic_mirror = "semrock-di03-r405-488-561-635-t1-25x36"
dichroic_515_mirror = "semrock-di03-r514-t1-25x36"
nilered_filter = "semrock-ff01-650-200"
shortpass_filter = "semrock-bsp01-785r"
longpass_filter_515 = "semrock-ff01-515-lp"

filters = [nilered_filter, dichroic_515_mirror, longpass_filter_515]
filter_spectra = S_F.get_dye_or_filter_data(
    names=filters, wavelength=wavelength, dye_or_filter=False
)

In [ ]:
folder = '/home/jbeckwith/Documents/pCloud/Chemistry/Lee/Code/Python/pyBayerSMLM/claude/Condition_1_NileRed'

In [ ]:
TM_Mode = SR_Functions.TemporalMedianMode(2)

fig, axs = SR_F.example_spots_singleframe(folder,
                                          smoothing_function=smoothing_function,
                                          gain_map=gain,
                                          offset_map=offset,
                                          rqe=rqe,
                                          read_noise=readnoise,
                                          variance=variance,
                                          temporal_median_mode=TM_Mode,
                                          ever_window=500,
                                          frame_index=250)

In [ ]:
TM_Mode = SR_Functions.TemporalMedianMode(0)

fig, axs = SR_F.example_spots_singleframe(folder,
                                          smoothing_function=smoothing_function,
                                          gain_map=gain,
                                          offset_map=offset,
                                          rqe=rqe,
                                          read_noise=readnoise,
                                          variance=variance,
                                          temporal_median_mode=TM_Mode,
                                          ever_window=500,
                                          frame_index=250)

In [ ]:
files = H_F.file_search(folder, '.tif', '')

In [ ]:
start_x, start_y, width, height = SR_F.helper.load_metadata_roi(
            folder, IO, use_fallback=True
        )

In [ ]:
masks = SR_F.mask.get_stacked_masks(
            start_x, start_y, width, height,
        )

# Crop calibration maps to ROI
cropped_maps = SR_F.helper.crop_calibration_maps(
    {
        "gain_map": gain,
        "offset_map": offset,
        "read_noise": readnoise,
        "rqe": rqe,
        "variance": variance,
    },
    start_x,
    start_y,
    width,
    height,
)

In [ ]:
test = IO.read_tiff(files[0], dtype="float32", frame=300)
test_pe = IO.convert_to_photoelectrons(test, gain_map=cropped_maps["gain_map"], offset_map=cropped_maps["offset_map"], rqe=cropped_maps["rqe"])

In [ ]:
vmin = np.percentile(test_pe, 0.01)
vmax = np.percentile(test_pe, 99.9)
plt.imshow(test_pe, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
plt.xlim([200, 300])
plt.ylim([200, 300])
plt.axis('off')
plt.colorbar()

In [ ]:

gain_map = cropped_maps["gain_map"]
offset_map = cropped_maps["offset_map"]
read_noise = cropped_maps["read_noise"]
rqe = cropped_maps["rqe"]
variance = cropped_maps["variance"]

In [ ]:
file_frame_counts = [SR_F.io.get_num_pages_in_TIF(f) for f in files]

In [ ]:
ever_window = 500

In [ ]:
frames_for_ever, center_idx = SR_F._load_frames_for_ever_window(
                files,
                0,  # First file
                300,
                ever_window,
                file_frame_counts,
            )

In [ ]:
ever_subtracted_adu_stack, ever_subtracted_pe_stack = (
                SR_F._compute_ever_background(
                    frames_for_ever,
                    window_size=ever_window,
                    spatial_filter_size=1,  # No spatial averaging for Bayer patterns
                    gain_map=gain_map,
                    offset_map=offset_map,
                )
            )
ever_subtracted_adu = ever_subtracted_adu_stack[center_idx]
ever_subtracted_pe = ever_subtracted_pe_stack[center_idx]

In [ ]:
vmin = np.percentile(ever_subtracted_pe, 0)
vmax = np.percentile(ever_subtracted_pe, 99)
plt.imshow(ever_subtracted_pe, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
plt.xlim([200, 300])
plt.ylim([200, 300])
plt.axis('off')
plt.colorbar()

In [ ]:
vmin = np.percentile(test, 1)
vmax = np.percentile(test, 99)
plt.imshow(test, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
plt.axis('off')
